In [1]:
import sys
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")
# Permite importar `proyecto` sin instalarlo: el código vive en src/
sys.path.insert(0, str(Path.cwd().resolve().parent / "src"))

# 03 · Preprocesamiento · de las decisiones del EDA a código con pruebas

El EDA terminó con 10 decisiones (`reports/01_eda.md`). Aquí se aplican con **funciones probadas** de
`src/proyecto/data/` (`cleaning.py` y `split.py`). Este notebook **solo orquesta**: llama a las funciones,
comprueba que se cumplen y guarda los datos.

Especificación que se implementa: `especificaciones/03-preprocesamiento.md`. Las decisiones y sus motivos están en
`reports/03_preprocesamiento.md`.

## 0. Cargar los datos crudos

`load_raw()` lee el CSV (separador `;`) y quita `GroupKey` en el primer paso. `data/raw/` no se modifica: guardamos su huella digital para comprobarlo al final.

In [2]:
import hashlib

import pandas as pd

from proyecto.config import GROUP_COL, LEAKAGE_COLUMNS, LISTING_KEY_COLUMNS, PROCESSED_DIR, RAW_CSV, TARGET
from proyecto.data.cleaning import clean, load_raw
from proyecto.data.split import assert_disjoint_members, split_by_member

def file_hash(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

raw_hash_before = file_hash(RAW_CSV)
raw = load_raw()
print(f"crudo: {raw.shape[0]:,} filas x {raw.shape[1]} columnas")

crudo: 113,937 filas x 80 columnas


## 1. Limpieza

`clean()` aplica las reglas en este orden: objetivo `is_default` (solo préstamos terminados) → una fila por
`ListingKey` → sin `BorrowerState` vacío → sin llaves ni ocupación → sin columnas de fuga → `IncomeRange` con
8 categorías fijas → valores imposibles a vacío → cuota 0 recalculada → fecha de originación → índice climático.

In [3]:
df = clean(raw)
print(f"antes: {raw.shape}  ->  después: {df.shape}")

antes: (113937, 80)  ->  después: (49552, 62)


## 2. Comprobaciones de la limpieza

Cada decisión es una **afirmación**: si alguna no se cumple, el notebook falla. Las cifras son las del EDA.

In [4]:
checks = {
    "49.552 préstamos terminados (cifra del EDA)": len(df) == 49_552,
    "15.380 impagos (31,04 %)": int(df[TARGET].sum()) == 15_380,
    "sin datos personales ni llaves": not set(LISTING_KEY_COLUMNS + ["GroupKey", "Occupation"]) & set(df.columns),
    "MemberKey conservado solo para partir": GROUP_COL in df.columns,
    "sin columnas de fuga": not set(LEAKAGE_COLUMNS + ["LoanStatus"]) & set(df.columns),
    "sin fechas dañadas": not {"ListingCreationDate", "DateCreditPulled"} & set(df.columns),
    "BorrowerState sin vacíos": df["BorrowerState"].notna().all(),
    "IncomeRange solo con categorías fijas": df["IncomeRange"].notna().all(),
    "sin puntaje de crédito 0": not (df["CreditScoreRangeLower"] == 0).any(),
    "sin ingreso mensual > 100.000": not (df["StatedMonthlyIncome"] > 100_000).any(),
    "sin cuota mensual 0": not (df["MonthlyLoanPayment"] == 0).any(),
    "fecha de originación es fecha": pd.api.types.is_datetime64_any_dtype(df["LoanOriginationDate"]),
    "índice climático entre 0 y 100": df["Precipitation_Risk_Index"].between(0, 100).all(),
}
result = pd.Series(checks, name="se cumple")
assert result.all(), result[~result]
result.to_frame()

,se cumple
49.552 préstamos terminados (cifra del EDA),True
"15.380 impagos (31,04 %)",True
sin datos personales ni llaves,True
MemberKey conservado solo para partir,True
sin columnas de fuga,True
sin fechas dañadas,True
BorrowerState sin vacíos,True
IncomeRange solo con categorías fijas,True
sin puntaje de crédito 0,True
sin ingreso mensual > 100.000,True


## 3. Valores atípicos: qué se tocó y qué no

Ninguna fila se borra por ser atípica. Lo **imposible** pasa a vacío (se imputa en la etapa 4, dentro del
`Pipeline`) y lo que se deduce con una regla exacta se repara. Lo extremo pero posible se conserva.

In [5]:
# Misma población antes de las reglas de valores (para contar qué cambió)
from proyecto.data.cleaning import build_target, drop_duplicate_listings, drop_missing_state

before = drop_missing_state(drop_duplicate_listings(build_target(raw)))
income_range = before["IncomeRange"].str.strip()
outliers = pd.DataFrame(
    {
        "filas": [
            (before["CreditScoreRangeLower"] == 0).sum(),
            (before["StatedMonthlyIncome"] > 100_000).sum(),
            ((before["StatedMonthlyIncome"] == 0) & ~income_range.isin(["$0", "Not employed"])).sum(),
            (before["MonthlyLoanPayment"] == 0).sum(),
            (before["DebtToIncomeRatio"] == 10.01).sum(),
            (before["EstimatedEffectiveYield"] < 0).sum(),
        ],
        "tratamiento": [
            "vacío (y su CreditScoreRangeUpper = 19)",
            "vacío",
            "vacío (0 con ingresos declarados)",
            "recalculada con la fórmula de cuota fija",
            "se conserva: es el tope del sistema y mantiene el orden",
            "se conserva: una pérdida estimada es posible",
        ],
    },
    index=[
        "CreditScoreRangeLower = 0",
        "StatedMonthlyIncome > 100.000",
        "StatedMonthlyIncome = 0 incoherente",
        "MonthlyLoanPayment = 0",
        "DebtToIncomeRatio = 10,01",
        "EstimatedEffectiveYield < 0",
    ],
)
outliers

,filas,tratamiento
CreditScoreRangeLower = 0,89,vacío (y su CreditScoreRangeUpper = 19)
StatedMonthlyIncome > 100.000,8,vacío
StatedMonthlyIncome = 0 incoherente,21,vacío (0 con ingresos declarados)
MonthlyLoanPayment = 0,838,recalculada con la fórmula de cuota fija
"DebtToIncomeRatio = 10,01",187,se conserva: es el tope del sistema y mantiene...
EstimatedEffectiveYield < 0,190,se conserva: una pérdida estimada es posible


**Lo que queda vacío** se imputa en la etapa 4 con estadísticas **solo del entrenamiento**. El vacío de 2005–2008 en las métricas de Prosper significa "no existía", no "perdido".

In [6]:
missing = df.isna().mean().loc[lambda s: s > 0].sort_values(ascending=False).rename("% vacío")
missing.to_frame().style.format("{:.1%}")

,% vacío
ScorexChangeAtTimeOfListing,79.0%
ProsperPaymentsOneMonthPlusLate,78.8%
TotalProsperLoans,78.8%
OnTimeProsperPayments,78.8%
ProsperPaymentsLessThanOneMonthLate,78.8%
ProsperPrincipalBorrowed,78.8%
ProsperPrincipalOutstanding,78.8%
TotalProsperPaymentsBilled,78.8%
CreditGrade,52.7%
ProsperRating (Alpha),47.6%


## 4. Partición por prestatario

**Unidad de análisis:** un préstamo personal de Prosper (una fila por `ListingKey`). **Entidad que se repite:**
el prestatario (`MemberKey`). Todos los préstamos de un prestatario van al mismo lado. La partición se
estratifica por `is_default`, con 20 % para prueba y semilla 42.

Partimos **una sola vez**. El conjunto de prueba se guarda y **no se mira** hasta la etapa 7.

In [7]:
train, test = split_by_member(df)
assert_disjoint_members(train, test)
assert len(train) + len(test) == len(df), "se perdieron filas al partir"

summary = pd.DataFrame(
    {
        "filas": [len(train), len(test)],
        "prestatarios": [train[GROUP_COL].nunique(), test[GROUP_COL].nunique()],
        "% de filas": [len(train) / len(df), len(test) / len(df)],
        "% impago": [train[TARGET].mean(), test[TARGET].mean()],
    },
    index=["entrenamiento", "prueba (cerrada)"],
)
summary.style.format({"filas": "{:,}", "prestatarios": "{:,}", "% de filas": "{:.1%}", "% impago": "{:.2%}"})

,filas,prestatarios,% de filas,% impago
entrenamiento,"39,641","32,341",80.0%,31.04%
prueba (cerrada),"9,911","8,111",20.0%,31.04%


### ¿Por qué importa partir por prestatario?

Lo comprobamos **solo dentro del entrenamiento**, porque la prueba no se toca. Comparamos dos formas de separar
una validación: por fila (los préstamos de un prestatario caen a ambos lados) y por prestatario. El modelo y los
datos son los mismos; solo cambia cómo se parte. Métrica: AUC-PR de la clase impago.

In [8]:
from lightgbm import LGBMClassifier
from sklearn.metrics import average_precision_score
from sklearn.model_selection import train_test_split

numeric = train.select_dtypes("number").drop(columns=[TARGET]).columns

def validation_auc_pr(fit_df, val_df):
    model = LGBMClassifier(n_estimators=200, learning_rate=0.05, random_state=42, verbose=-1)
    model.fit(fit_df[numeric], fit_df[TARGET])
    return average_precision_score(val_df[TARGET], model.predict_proba(val_df[numeric])[:, 1])

# A) por fila
row_fit, row_val = train_test_split(train, test_size=0.2, stratify=train[TARGET], random_state=42)
# B) por prestatario
member_fit, member_val = split_by_member(train)

pd.DataFrame(
    {
        "prestatarios en ambos lados": [len(set(row_fit[GROUP_COL]) & set(row_val[GROUP_COL])), 0],
        "AUC-PR validación": [validation_auc_pr(row_fit, row_val), validation_auc_pr(member_fit, member_val)],
    },
    index=["por fila (mal)", "por prestatario (bien)"],
).round(3)

,prestatarios en ambos lados,AUC-PR validación
por fila (mal),2013,0.620
por prestatario (bien),0,0.598


**Lectura.** Esto es solo una demostración rápida (variables numéricas, sin ajuste), no el modelo del proyecto.
Si la partición por fila da un AUC-PR mayor, esa ventaja es **falsa**: el modelo vio préstamos del mismo
prestatario al entrenar. En producción llegan prestatarios nuevos, y el rendimiento honesto es el de la fila de abajo.
Con solo ~14 % de prestatarios repetidos en la población, la diferencia puede ser pequeña; la regla se mantiene
porque no depende del tamaño del efecto.

## 5. Guardar los datos de la etapa

Se guardan en `data/processed/` en formato parquet (conserva fechas y categorías). No se versionan: se regeneran con este notebook.

In [9]:
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
train.to_parquet(PROCESSED_DIR / "train.parquet", index=False)
test.to_parquet(PROCESSED_DIR / "test.parquet", index=False)

assert file_hash(RAW_CSV) == raw_hash_before, "data/raw cambió"
print("guardado en data/processed/: train.parquet y test.parquet (cerrada hasta la etapa 7)")
print("data/raw/ intacta")

guardado en data/processed/: train.parquet y test.parquet (cerrada hasta la etapa 7)
data/raw/ intacta


## 6. Las pruebas automáticas

Desde la raíz del repo:

```bash
pytest -q
```

`tests/test_cleaning.py` tiene una prueba por regla (caso normal y caso sucio hechos a mano) y
`tests/test_split.py` verifica que ningún `MemberKey` cae en dos particiones y que no se pierden filas.

## Entregable de esta etapa (hito H1)

- [x] La limpieza corre de principio a fin desde el repo.
- [x] La partición respeta la unidad de análisis (`MemberKey`) y está **verificada con una prueba**.
- [x] `pytest` en verde.
- [x] `data/raw/` no cambió (huella digital comprobada arriba).
- [x] Unidad de análisis declarada por escrito (sección 4).

**Siguiente:** `04_features.ipynb`.